# Demucs / HTDemucs 구조 분석 노트북
### SongLingo 기술 탐구 · 2026-11-06

**탐구 질문:** Demucs는 혼합 음악에서 어떻게 vocals·drums·bass·other를 분리하는가? HTDemucs의 waveform/spectrogram branch, CNN Encoder–Decoder, Transformer를 실제 소스 코드와 텐서 흐름으로 확인한다.

이 노트북은 `greenkey20/song-lingo`의 `docs/프로젝트_안내_및_주제_검토.md` 15.4절과 프로젝트의 `separate.py`를 출발점으로 삼는다.

## 0. 분석 대상: 어느 저장소를 봐야 하나?

질문에 대한 답은 **둘 다 보되, 목적이 다르다**는 거야.

| 저장소 | 분석 목적 |
|---|---|
| [facebookresearch/demucs](https://github.com/facebookresearch/demucs) | 모델 자체의 구조·레이어·forward·추론 알고리즘 분석 |
| [greenkey20/song-lingo](https://github.com/greenkey20/song-lingo) | 우리 서비스가 모델을 어떻게 로드·전처리·호출·후처리하는지 분석 |

따라서 “Demucs 구조 분석”의 중심은 원본 Demucs 저장소의 `HTDemucs` 구현이 맞아. 하지만 모델 구조만 읽으면 SongLingo에서 실제로 어떤 입력을 주고 어떤 출력을 사용하는지 놓치므로 `separate.py`도 함께 추적한다.

**분석 원칙**
- 15.4절의 파라미터·성능 수치는 해당 프로젝트 컴퓨터에서 확인한 값이지, 모든 버전에 적용되는 상수가 아니다.
- 개념적 설명과 설치된 패키지에서 확인한 구현 사실을 구분한다.
- “분리하면 STT가 좋아진다”는 결론을 미리 가정하지 않는다.


## 1. 문제 정의: 음악 source separation

혼합 음악은 여러 소스가 합쳐진 신호다.

\[
x(t) = s_{vocals}(t) + s_{drums}(t) + s_{bass}(t) + s_{other}(t)
\]

Demucs는 혼합 신호를 입력받아 학습된 모델로 각 소스의 추정값을 출력한다.

SongLingo에서 vocals를 사용하는 목적은 ① Whisper가 반주 없이 가사를 인식하도록 비교하고 ② 분리된 보컬을 후속 음성 처리의 입력 후보로 쓰는 것이다.

**중요한 실험적 배경:** 15.4절의 8개 구간 비교에서는 vocals의 micro WER(0.606)이 원곡(0.506)보다 전체적으로 나빴다. 따라서 분리는 가설이지, 항상 유효한 전처리가 아니다.


## 2. SongLingo의 호출 경로

`separate.py`의 핵심 흐름:

1. `load_clip()` — 입력 오디오를 44.1 kHz 스테레오 `(2, samples)`로 변환
2. `_load_model()` — `get_model("htdemucs")`, `eval()`
3. `separate_vocals()` — NumPy 배열을 Tensor로 변환하고 배치 축 추가
4. 채널 평균 신호의 평균·표준편차로 정규화
5. `apply_model(..., split=True, overlap=0.25)` 호출
6. 정규화 복원 후 `model.sources`에서 vocals 출력 선택

프로젝트 루트에서 실행한다. 경로가 다르면 `PROJECT_ROOT`를 수정한다.


In [ ]:
from pathlib import Path
import sys, platform, inspect

PROJECT_ROOT = Path.cwd()  # 필요하면 Path("/path/to/song-lingo")로 수정
print("Python:", sys.version)
print("Platform:", platform.platform())
print("separate.py:", (PROJECT_ROOT / "separate.py").exists())


In [ ]:
# 프로젝트의 전처리·모델 호출 코드 확인
p = PROJECT_ROOT / "separate.py"
if p.exists():
    lines = p.read_text(encoding="utf-8").splitlines()
    for start, end in [(1, 35), (65, 125)]:
        print(f"\\n--- lines {start}-{end} ---")
        print("\\n".join(f"{i+1:4}: {v}" for i, v in enumerate(lines[start-1:end], start-1)))
else:
    print("PROJECT_ROOT를 song-lingo 루트로 설정해.")


### 확인 질문
- `separate.py`는 모델 내부를 구현하는가, 아니면 사전학습 모델을 호출하는가?
- `(2, samples)`와 `(1, 2, samples)`의 각 축은 무엇을 의미하는가?
- `model.eval()`과 `torch.no_grad()`는 같은 기능인가?


## 3. 실행 환경과 모델 메타데이터

15.4절에 기록된 프로젝트 환경의 확인값:

| 항목 | 당시 확인값 |
|---|---:|
| 패키지 | `demucs 4.1.0` |
| 모델 | `htdemucs` |
| 내부 클래스 | `HTDemucs` (외부 wrapper `BagOfModels`) |
| 파라미터 | 약 42.0M |
| depth / channels | 4 / 48 |
| nfft / hop_length | 4096 / 1024 |
| 입력 | 44.1 kHz stereo |
| segment | 약 7.8초 |
| 가중치 | 약 80 MB |

**주의:** `get_model()`은 체크포인트가 캐시에 없으면 다운로드할 수 있다. 아래 `LOAD_MODEL`을 `True`로 바꾸기 전 네트워크 다운로드 여부를 고려해. 프로젝트에서 Whisper와 Demucs를 같은 커널에 로드할 때 OpenMP 충돌로 커널이 종료된 적이 있으므로 이 노트북에서는 Whisper를 import하지 않는다.


In [ ]:
try:
    import demucs
    print("demucs module:", demucs.__file__)
    print("version:", getattr(demucs, "__version__", "version attribute unavailable"))
except Exception as e:
    print("Demucs import error:", repr(e))


In [ ]:
# 모델 로딩은 명시적으로 opt-in
LOAD_MODEL = False  # 준비되었을 때 True
bag = inner = None

if LOAD_MODEL:
    import torch
    from demucs.pretrained import get_model
    bag = get_model("htdemucs")
    bag.eval()
    inner = bag.models[0] if hasattr(bag, "models") else bag
    print("wrapper:", type(bag).__name__)
    print("inner:", type(inner).__name__)
    print("sources:", bag.sources)
    print("samplerate:", bag.samplerate)
    print("audio_channels:", bag.audio_channels)
    print("segment:", getattr(inner, "segment", None))
    n = sum(p.numel() for p in inner.parameters())
    print(f"parameters: {n:,} ({n/1e6:.2f} M)")
else:
    print("LOAD_MODEL=False: model loading skipped")


## 4. HTDemucs의 핵심: Hybrid

`HT`는 Hybrid Transformer를 가리킨다. 15.4절에서 해당 설치 모델의 구조로 확인한 요소는 다음과 같다.

- **Spectrogram branch:** STFT 기반 시간–주파수 표현. 문서에서 `encoder` / `decoder`로 확인.
- **Waveform branch:** 파형을 직접 처리. 문서에서 `tencoder` / `tdecoder`로 확인.
- **Cross Transformer:** 두 표현 사이의 정보를 교환. 문서에서 `crosstransformer`로 확인.

파형은 시간 영역의 세밀한 신호를 직접 담고, 스펙트로그램은 주파수 성분의 시간적 변화를 표현한다. 두 관점을 결합하는 것이 하이브리드 구조의 직관이다.

단, 각 branch의 역할을 지나치게 단정하지 말자. 실제로 무엇을 계산하는지는 `forward()`와 각 모듈의 입력·출력 shape으로 검증해야 한다.


### 4.1 STFT와 모델 설정 연결

\[
X(m,k)=\sum_{n=0}^{N-1}x[n+mH]w[n]e^{-j2\pi kn/N}
\]

`N`은 FFT window 길이, `H`는 hop length다. 15.4절에 기록된 `nfft=4096`, `hop_length=1024`, `sr=44100`을 이용해 시간·주파수 해상도를 계산한다.


In [ ]:
sr, nfft, hop = 44_100, 4096, 1024
print(f"FFT bin spacing: {sr/nfft:.3f} Hz")
print(f"Window duration: {nfft/sr*1000:.2f} ms")
print(f"Hop duration: {hop/sr*1000:.2f} ms")


## 5. 원본 Demucs 소스 코드 추적

분석 중심은 설치된 패키지의 실제 `HTDemucs` 클래스다. 설치 버전에 따라 파일·속성명이 달라질 수 있으므로 소스 경로와 시그니처부터 확인한다.

살펴볼 대상:
- `demucs.htdemucs.HTDemucs`
- `demucs.apply.apply_model`
- `demucs.pretrained.get_model`
- Transformer 관련 구현


In [ ]:
if LOAD_MODEL:
    from demucs.apply import apply_model
    print("HTDemucs file:", inspect.getsourcefile(type(inner)))
    print("apply_model file:", inspect.getsourcefile(apply_model))
    print("constructor:", inspect.signature(type(inner)))
else:
    print("LOAD_MODEL=True 후 실행")


In [ ]:
if LOAD_MODEL:
    for name in ["sources", "audio_channels", "samplerate", "segment",
                 "depth", "channels", "nfft", "hop_length", "cac"]:
        if hasattr(inner, name):
            print(name, "=", getattr(inner, name))
    print("\\nTop-level child modules:")
    for name, module in inner.named_children():
        print(f"{name:24} {type(module).__name__}")
else:
    print("LOAD_MODEL=True 후 실행")


In [ ]:
# 클래스 전체 구현을 읽는다. 길면 출력 일부씩 확인한다.
if LOAD_MODEL:
    src = inspect.getsource(type(inner))
    print(src[:30000])
else:
    print("LOAD_MODEL=True 후 실행")


### 소스 코드 읽기 체크리스트

| 코드 위치 | 확인할 내용 |
|---|---|
| `__init__` | 입력 채널, depth, FFT, segment 설정 |
| `encoder` / `decoder` | spectrogram branch의 feature map 변화 |
| `tencoder` / `tdecoder` | waveform branch의 텐서 변화 |
| `crosstransformer` | 두 branch 사이 정보 교환 방식 |
| `forward()` | 실제 모듈 호출 순서와 최종 출력 shape |
| `apply_model()` | segment 분할, overlap, 결과 결합 |

클래스 속성 이름만 보고 동작을 추측하지 말고 `forward()`의 호출 순서를 따라간다.


## 6. Tensor shape 관찰: forward hook

모델 구조를 이해하는 데 가장 직접적인 증거 중 하나는 레이어 입출력 텐서의 shape이다.

SongLingo 입력은 `(channels, samples)`, 모델 입력은 배치 축을 추가한 `(batch, channels, samples)` 형태다. 아래 코드는 상위 모듈에 hook을 등록한다. 실제 forward 실험은 적절한 44.1 kHz 스테레오 입력과 충분한 길이를 준비한 뒤 별도로 수행한다.


In [ ]:
from collections import OrderedDict
activation_shapes = OrderedDict()
hooks = []

def shape_of(x):
    if hasattr(x, "shape"):
        return tuple(x.shape)
    if isinstance(x, (tuple, list)):
        return [shape_of(v) for v in x]
    if isinstance(x, dict):
        return {k: shape_of(v) for k, v in x.items()}
    return type(x).__name__

def make_hook(name):
    def hook(module, inputs, output):
        activation_shapes[name] = {
            "class": type(module).__name__,
            "input": shape_of(inputs),
            "output": shape_of(output)
        }
    return hook

if LOAD_MODEL:
    for name, module in inner.named_children():
        hooks.append(module.register_forward_hook(make_hook(name)))
    print("registered hooks:", len(hooks))
else:
    print("LOAD_MODEL=True 후 실행")


In [ ]:
# 실험용 입력 준비 예시. AUDIO_PATH를 본인 로컬 파일로 지정한다.
# 실행 전 입력이 44.1 kHz stereo인지 확인한다.
AUDIO_PATH = None  # 예: PROJECT_ROOT / "data/song_work/clip_mix.wav"
RUN_FORWARD = False

if LOAD_MODEL and RUN_FORWARD and AUDIO_PATH:
    import numpy as np
    import torch
    import audio
    x, input_sr, _ = audio.load_raw(AUDIO_PATH)  # (samples, channels)
    if input_sr != 44_100:
        raise ValueError(f"expected 44100 Hz, got {input_sr}")
    x = x.T
    if x.shape[0] == 1:
        x = np.repeat(x, 2, axis=0)
    wav = torch.from_numpy(np.ascontiguousarray(x, dtype=np.float32))[None]
    activation_shapes.clear()
    with torch.no_grad():
        y = inner(wav)
    print("input:", tuple(wav.shape), "output:", tuple(y.shape))
    for name, info in activation_shapes.items():
        print(name, info)
else:
    print("forward pass skipped; configure AUDIO_PATH and RUN_FORWARD")


In [ ]:
# 사용 후 hook 제거
for h in hooks:
    h.remove()
hooks.clear()
print("hooks removed")


## 7. `split=True`, `overlap=0.25` 추론

SongLingo는 `apply_model(..., split=True, overlap=0.25)`를 사용한다.

- 긴 입력을 segment 단위로 나누어 처리한다.
- 인접 segment가 겹치도록 해 경계 artifact를 줄이려는 목적이다.
- 실제 분할 길이·겹침 길이·결과 결합 방식은 설치된 `demucs.apply.apply_model` 구현으로 확인한다.

아래 셀은 해당 함수의 소스를 출력한다.


In [ ]:
if LOAD_MODEL:
    from demucs.apply import apply_model
    apply_src = inspect.getsource(apply_model)
    print(apply_src[:25000])
else:
    print("LOAD_MODEL=True 후 실행")


## 8. 음향 실험: mixture / vocals / residual

15.4절의 11-04 추가 메모에서 제안한 실험이다.

**가설:** 보컬과 악기의 주파수 범위는 많이 겹친다. 따라서 “이 대역은 보컬, 저 대역은 악기”라는 고정 필터만으로 완전한 분리를 하기 어렵다.

비교할 신호:
- `mixture`: 원곡 구간
- `vocals`: Demucs가 추정한 보컬
- `residual`: mixture − vocals

Residual은 단순 차분이다. 나머지 3개 stem 합계와 정확히 일치하는지는 정렬·수치 오차 등을 확인한다. 스펙트럼 시각화는 분리 품질의 절대 점수가 아니다.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import stft

# 실제 파일을 읽어 아래 변수에 (channels, samples) 배열을 넣는다.
mixture = None
vocals = None

# 예:
# mix, sr, _ = audio.load_raw(PROJECT_ROOT / "data/song_work/clip_mix.wav")
# voc, sr2, _ = audio.load_raw(PROJECT_ROOT / "data/song_work/clip_vocals.wav")
# mixture, vocals = mix.T, voc.T

def as_mono(x):
    x = np.asarray(x)
    return x.mean(axis=0) if x.ndim == 2 else x

def show_spectrograms(mixture, vocals, sr=44_100):
    xm, xv = as_mono(mixture), as_mono(vocals)
    n = min(len(xm), len(xv))
    signals = [xm[:n], xv[:n], xm[:n] - xv[:n]]
    titles = ["Mixture", "Estimated vocals", "Residual = mixture - vocals"]
    fig, axes = plt.subplots(3, 1, figsize=(14, 9), constrained_layout=True)
    for ax, sig, title in zip(axes, signals, titles):
        f, t, z = stft(sig, fs=sr, nperseg=4096, noverlap=3072)
        db = 20*np.log10(np.abs(z) + 1e-7)
        im = ax.pcolormesh(t, f, db, shading="auto")
        ax.set_ylim(0, 10000)
        ax.set_title(title)
        ax.set_ylabel("Hz")
        fig.colorbar(im, ax=ax, label="dB")
    axes[-1].set_xlabel("Time (s)")
    plt.show()

if mixture is not None and vocals is not None:
    show_spectrograms(mixture, vocals)
else:
    print("mixture와 vocals를 준비한 뒤 실행")


### 8.1 주파수 대역별 에너지 비율

대역별 `vocals energy / mixture energy`를 계산해 볼 수 있다. 이는 보컬 추정 신호가 원곡 에너지에 비해 어느 정도의 에너지를 갖는지 보는 **기술적 관찰값**이다.

위상·상관관계 때문에 이를 “원곡에서 보컬이 차지하는 실제 에너지 비율”이라고 단정하면 안 된다. 정답 stem이 없으면 분리의 절대 정확도를 평가할 수도 없다.


In [ ]:
def band_energy_ratio(mixture, vocals, sr=44_100):
    bands = [(0,250),(250,500),(500,1000),(1000,2000),
             (2000,4000),(4000,8000),(8000,12000),(12000,sr/2)]
    xm, xv = as_mono(mixture), as_mono(vocals)
    n = min(len(xm), len(xv))
    f, _, zm = stft(xm[:n], fs=sr, nperseg=4096, noverlap=3072)
    _, _, zv = stft(xv[:n], fs=sr, nperseg=4096, noverlap=3072)
    pm, pv = np.abs(zm)**2, np.abs(zv)**2
    for lo, hi in bands:
        mask = (f >= lo) & (f < hi)
        em, ev = pm[mask].sum(), pv[mask].sum()
        print(f"{lo:5}-{hi:5} Hz | vocals={ev:.3e} mixture={em:.3e} ratio={ev/em if em else np.nan:.3f}")

if mixture is not None and vocals is not None:
    band_energy_ratio(mixture, vocals)
else:
    print("실제 mixture/vocals를 준비한 뒤 실행")


## 9. 대조군: 고정 대역통과 필터

“주파수 대역만으로 분리할 수 있는가?”를 확인하려면 간단한 band-pass filter를 대조군으로 둘 수 있다. 같은 원곡 구간에 필터를 적용하고 Demucs vocals와 청취 및 동일 STT 조건으로 비교한다.

필터 cutoff는 결과를 본 뒤 유리하게 바꾸지 말고 실험 전에 정한다. 필터는 선택한 대역 밖의 보컬 정보도 제거할 수 있다.


In [ ]:
from scipy.signal import butter, sosfiltfilt

def bandpass_filter(x, sr=44_100, low=150, high=5000, order=4):
    if not (0 < low < high < sr/2):
        raise ValueError("require 0 < low < high < Nyquist")
    sos = butter(order, [low, high], btype="bandpass", fs=sr, output="sos")
    return sosfiltfilt(sos, x)

if mixture is not None:
    filtered = bandpass_filter(as_mono(mixture))
    print("filtered samples:", filtered.shape)
else:
    print("mixture를 준비한 뒤 실행")


## 10. 평가: STT WER과 음원 분리 품질은 다르다

| 평가 관점 | 방법 | 답하는 질문 |
|---|---|---|
| SongLingo 목적 | WER / CER | 분리 후 STT가 정답 가사에 가까워졌나? |
| 분리 자체 | SDR / SI-SDR | 추정 stem이 기준 stem에 얼마나 가까운가? |
| 처리 비용 | latency / real-time factor | 실제 사용에 필요한 시간은? |
| 청취 | 주관 평가 | 반주 잔류·보컬 손실·artifact가 들리는가? |

SDR/SI-SDR에는 정답 vocals stem이 필요하다. Demucs가 생성한 vocals를 자기 자신의 정답으로 삼으면 평가가 성립하지 않는다.

**프로젝트 문서의 관찰:** CPU 20초 분리에 약 65초가 걸린 사례, WMA 32 kHz·약 33 kbps 입력, 분리 후 STT가 악화된 구간이 기록되어 있다. 이는 해당 데이터·환경의 결과이지 Demucs 전체 성능에 대한 일반 결론은 아니다.


## 11. 강사님께 질문할 내용

실제 코드를 읽다가 답을 찾지 못한 항목만 남긴다.

1. HTDemucs의 두 branch는 학습 과정에서 서로 보완적인 표현을 어떻게 학습하는가?
2. `crosstransformer`에 들어가는 waveform/spectrogram 표현은 어떤 shape으로 token화되는가?
3. CNN Encoder–Decoder의 skip connection은 source reconstruction에서 어떤 역할을 하는가?
4. `cac=True`는 복소수 스펙트로그램을 어떤 표현으로 다루는가?
5. SongLingo의 목적에는 WER/CER 비교가 실용 평가로 충분한가? 분리 자체의 품질을 주장하려면 SDR/SI-SDR 실험이 필요한가?


## 12. 실험 기록 템플릿

| 항목 | 기록 |
|---|---|
| 날짜 / 실험 ID | |
| Demucs 버전 / 모델 | |
| 익명 입력 ID / 구간 길이 | |
| 입력 SR / 채널 | |
| 실행 장치 / thread 수 | |
| mixture 청취 평가 | |
| vocals 청취 평가 | |
| residual 청취 평가 | |
| 처리 시간 | |
| STT 모델 / 설정 | |
| WER / CER (mixture) | |
| WER / CER (vocals) | |
| 관찰·해석·한계 | |
| 다음 질문 | |

**데이터 취급:** 프로젝트 문서의 원칙대로 곡 파일·파생 오디오·스펙트로그램은 로컬 `data/` 또는 `results/`에 두고 공개 저장소에 커밋하지 않는다.


## 13. 완료 기준과 참고 자료

### 완료 기준
- [ ] 원본 Demucs 저장소와 SongLingo 통합 코드의 역할 차이를 설명한다.
- [ ] 설치된 `HTDemucs` 클래스의 실제 소스 경로와 핵심 모듈을 확인한다.
- [ ] `forward()`의 호출 순서를 추적한다.
- [ ] 주요 모듈의 입출력 텐서 shape을 최소 한 번 기록한다.
- [ ] `apply_model(split=True, overlap=0.25)`의 분할·결합 방식을 설명한다.
- [ ] mixture/vocals/residual 스펙트로그램을 동일 설정으로 비교한다.
- [ ] 분리 전후 WER 결과와 한계를 설명한다.

### 참고
- [facebookresearch/demucs — GitHub](https://github.com/facebookresearch/demucs)
- [SongLingo 프로젝트 문서 15.4절](https://github.com/greenkey20/song-lingo/blob/main/docs/프로젝트_안내_및_주제_검토.md)
- [SongLingo `separate.py`](https://github.com/greenkey20/song-lingo/blob/main/separate.py)
- [Hybrid Spectrogram and Waveform Source Separation](https://arxiv.org/abs/2211.08553)

논문·저장소에서 확인한 내용과 이 프로젝트 환경에서 확인한 구현값을 발표에서 명확히 구분한다.
